# Branches, tags and table maintenance

Validate on a branch, rehearse promotion and rollback, then measure and maintain a table whose data files are owned only by Iceberg.

![Branches, tags and table maintenance](assets/07_flow.png)


## Connect the notebook to Spark

Run from top to bottom in a fresh Python kernel. Close the previous Spark session first.
Connection values use the environment when present, otherwise the Kubernetes Service names below.
Credentials stay in the existing course environment. No extra workbench manifest update is needed.
Short Service names resolve within the Pod's namespace; use a fully qualified Service name when crossing namespaces.
A Service name stays stable when its Pods are replaced. A container name or Pod IP is not the connection contract.

The notebook kernel is the **driver**. `.master(master_url)` connects this application to the
standalone Spark master. The master allocates executors on its workers. Workers must also reach
the notebook driver address and its ports. The Spark services and workbench run inside Kubernetes;
this is a standalone Spark application, rather than native Kubernetes submission.

No separate `spark-submit` command is needed while running notebook cells. Read the master URL
from the environment and inspect `spark.sparkContext.master` after the connection is created.


In [ ]:
# Import the standard Python and Spark classes.
import os
import re
import socket
import uuid

from datetime import date, datetime
from decimal import Decimal

from pyspark.sql import SparkSession


In [ ]:
# Use environment overrides or the stable Kubernetes Service names.
master_url = os.environ.get('SPARK_MASTER_URL', 'spark://spark-master:7077')
driver_host = os.environ.get('SPARK_DRIVER_HOST', 'jupyter')

hdfs_uri = os.environ.get('HDFS_URI', 'hdfs://namenode:9000').rstrip('/')
metastore_uri = os.environ.get('HIVE_METASTORE_URI', 'thrift://hive-metastore:9083')

polaris_base_url = os.environ.get('POLARIS_BASE_URL', 'http://polaris:8181').rstrip('/')
polaris_uri = os.environ.get('POLARIS_URI', polaris_base_url + '/api/catalog').rstrip('/')
polaris_warehouse = os.environ.get('POLARIS_WAREHOUSE', os.environ.get('POLARIS_CATALOG', 'training'))
s3_endpoint = os.environ.get('S3_ENDPOINT', 'http://seaweedfs:8333')
storage_region = os.environ.get('AWS_DEFAULT_REGION', os.environ.get('AWS_REGION', 'us-east-1'))
polaris_realm = os.environ.get('POLARIS_REALM', 'POLARIS')

assert master_url.startswith('spark://'), 'Use the course standalone Spark master URL'


In [ ]:
# Display the resolved connection values before Spark starts.
resolved_connections = {
    'Spark master': master_url,
    'Notebook driver': driver_host,
    'HDFS filesystem': hdfs_uri,
    'Hive metastore': metastore_uri,
    'Polaris REST catalog': polaris_uri,
    'Polaris catalog name': polaris_warehouse,
    'Polaris realm': polaris_realm,
    'S3 endpoint': s3_endpoint,
    'Storage region': storage_region,
}

for setting, value in resolved_connections.items():
    print(f'{setting:24} = {value}')


In [ ]:
# Read existing credentials without printing their values.
polaris_credential = os.environ.get('POLARIS_CREDENTIAL')
if not polaris_credential:
    polaris_credential = (
        os.environ['POLARIS_CLIENT_ID'] + ':' + os.environ['POLARIS_CLIENT_SECRET']
    )

s3_access_key = os.environ['AWS_ACCESS_KEY_ID']
s3_secret_key = os.environ['AWS_SECRET_ACCESS_KEY']


### Give this execution its own tables and files

`ns` is the database or namespace used by the table names below. `root` is this execution's HDFS directory.
A new run ID keeps a full rerun separate from earlier work. Keep the printed values if you want to inspect the results later.


In [ ]:
# Choose isolated names for this notebook execution.
run_id = os.environ.get('LESSON_RUN_ID', uuid.uuid4().hex[:10])
assert re.fullmatch(r'[a-z0-9_]{1,32}', run_id), 'Use a lowercase execution ID'

ns = f'migration11_07_{run_id}'
root = hdfs_uri + f'/workshop/11-iceberg-migration/07/{run_id}'

print('Namespace:', ns)
print('HDFS directory:', root)


In [ ]:
# Make the warehouse locations and Polaris catalog selection explicit.
hive_warehouse = root + '/managed'
hadoop_iceberg_warehouse = root + '/iceberg-warehouse'
hive_iceberg_warehouse = root + '/iceberg-hive-warehouse'

print('Hive managed warehouse:      ', hive_warehouse)
print('HadoopCatalog warehouse:     ', hadoop_iceberg_warehouse)
print('Hive Iceberg warehouse:      ', hive_iceberg_warehouse)
print('Polaris warehouse parameter: ', polaris_warehouse, '(catalog name, not an S3 path)')


### Configure the driver and executors

`.master(master_url)` selects the cluster. The driver address is different: executors use it to call back to this notebook.
Driver ports must be reachable from the worker network. A small executor allocation is sufficient for these datasets.


In [ ]:
# Connect the application to the master and advertise the notebook driver.
builder = (
    SparkSession.builder
    .appName('iceberg-migration-07')
    .master(master_url)
    .config('spark.driver.host', driver_host)
    .config('spark.driver.bindAddress', '0.0.0.0')
    .config('spark.driver.port', '7078')
    .config('spark.blockManager.port', '7079')
)


In [ ]:
# Set the resource limits, time zone and credential redaction.
builder = (
    builder
    .config('spark.cores.max', '1')
    .config('spark.executor.cores', '1')
    .config('spark.executor.memory', '512m')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.sql.adaptive.enabled', 'false')
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.openlineage.disabled', 'true')
    .config('spark.redaction.regex', '(?i)secret|password|token|access[.]?key|credential')
)


### Configure the catalogs

`spark_catalog` reads Hive tables and supports in-place conversion through the metastore.
`lake` is a HadoopCatalog: its catalog metadata and data live on HDFS, without the Hive metastore.
`iceberg` is a HiveCatalog: table registration lives in the Hive metastore, with files on HDFS.
`polaris` is a REST catalog for Iceberg tables in the object-store warehouse.
Catalog configuration is required before creating the Spark session.


In [ ]:
# Configure the default HDFS filesystem and managed-table warehouse.
builder = (
    builder
    .config('spark.hadoop.fs.defaultFS', hdfs_uri)
    .config('spark.hadoop.dfs.client.use.datanode.hostname', 'true')
    .config('spark.sql.warehouse.dir', hive_warehouse)
)


In [ ]:
# Enable Iceberg SQL and keep the Hive metastore available.
builder = (
    builder
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.hadoop.hive.metastore.uris', metastore_uri)
    .config('spark.sql.catalog.spark_catalog', 'org.apache.iceberg.spark.SparkSessionCatalog')
    .config('spark.sql.catalog.spark_catalog.type', 'hive')
    .config('spark.sql.catalog.spark_catalog.uri', metastore_uri)
    .config('spark.sql.catalog.spark_catalog.warehouse', hive_warehouse)
)


In [ ]:
# Keep the HDFS Iceberg warehouse under this execution directory.
builder = (
    builder
    .config('spark.sql.catalog.lake', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.lake.type', 'hadoop')
    .config('spark.sql.catalog.lake.warehouse', hadoop_iceberg_warehouse)
)


In [ ]:
# Register a Hive-metastore Iceberg catalog with HDFS storage.
builder = (
    builder
    .config('spark.sql.catalog.iceberg', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.iceberg.type', 'hive')
    .config('spark.sql.catalog.iceberg.uri', metastore_uri)
    .config('spark.sql.catalog.iceberg.warehouse', hive_iceberg_warehouse)
)


### Configure both S3 clients

Spark reads and writes ordinary `s3a://` files through Hadoop S3A.
Iceberg's Polaris tables use `s3://` locations through S3FileIO. These are separate clients;
configure the endpoint, credentials and path-style access for both.
HTTP disables S3A TLS only for this course endpoint. HTTPS keeps TLS enabled.


In [ ]:
# Configure Hadoop S3A for ordinary Spark object-store files.
builder = (
    builder
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.endpoint', s3_endpoint)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', str(s3_endpoint.startswith('https://')).lower())
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.hadoop.fs.s3a.access.key', s3_access_key)
    .config('spark.hadoop.fs.s3a.secret.key', s3_secret_key)
)


In [ ]:
# Connect the Polaris REST catalog using credentials from the environment.
builder = (
    builder
    .config('spark.sql.catalog.polaris', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.polaris.type', 'rest')
    .config('spark.sql.catalog.polaris.uri', polaris_uri)
    .config('spark.sql.catalog.polaris.warehouse', polaris_warehouse)
    .config('spark.sql.catalog.polaris.oauth2-server-uri', polaris_uri + '/v1/oauth/tokens')
    .config('spark.sql.catalog.polaris.credential', polaris_credential)
    .config('spark.sql.catalog.polaris.scope', 'PRINCIPAL_ROLE:ALL')
    .config('spark.sql.catalog.polaris.header.Polaris-Realm', polaris_realm)
    .config('spark.sql.catalog.polaris.rest-metrics-reporting-enabled', 'false')
)


In [ ]:
# Let Iceberg read and write the configured S3 warehouse.
builder = (
    builder
    .config('spark.sql.catalog.polaris.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.polaris.s3.endpoint', s3_endpoint)
    .config('spark.sql.catalog.polaris.s3.path-style-access', 'true')
    .config('spark.sql.catalog.polaris.s3.access-key-id', s3_access_key)
    .config('spark.sql.catalog.polaris.s3.secret-access-key', s3_secret_key)
    .config('spark.sql.catalog.polaris.client.region', storage_region)
)


In [ ]:
# Start Spark and display the master actually used.
spark = builder.enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

print('Spark version:', spark.version)
print('Connected master:', spark.sparkContext.master)

assert spark.sparkContext.master == master_url, 'Spark must use the configured master'


### Verify that work reaches an executor

The following small task runs on a worker. Its process hostname must differ from the notebook driver's hostname.
This checks that the application is using a separate executor instead of silently running in local mode.


In [ ]:
# Verify distributed execution before writing course data.
worker_names = (
    spark.sparkContext
    .parallelize([1], 1)
    .map(lambda _: __import__('socket').gethostname())
    .collect()
)

assert all(name != socket.gethostname() for name in worker_names), 'Use a separate executor'
print('PASS: Distributed Spark executor verified')


In [ ]:
# Put managed Hive data under this execution directory.
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS spark_catalog.{ns}
    LOCATION '{root}/managed'
""")


In [ ]:
# Create the namespace for HDFS Iceberg exercises.
spark.sql(f"""
    CREATE NAMESPACE IF NOT EXISTS lake.{ns}
""")


In [ ]:
# Use the existing Hive database through the named Iceberg catalog.
spark.sql(f"""
    CREATE NAMESPACE IF NOT EXISTS iceberg.{ns}
""")


In [ ]:
# Create the namespace for object-store Iceberg exercises.
spark.sql(f"""
    CREATE NAMESPACE IF NOT EXISTS polaris.{ns}
""")


### Inspect HDFS files directly

These are Hadoop's filesystem objects. They are used only to verify file existence or to prepare the scoped orphan-file exercise.


In [ ]:
# Use the configured HDFS filesystem for file checks.
hdfs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
HadoopPath = spark._jvm.org.apache.hadoop.fs.Path


## Stage the source data in HDFS
These six orders span three days. Amounts use fixed precision and all timestamps use UTC.
The explicit `order_date` column represents a traditional Hive partition key.


In [ ]:
# Define the staged rows and their explicit column types.
staged_orders_df = spark.createDataFrame(
    [
        (101, 1, datetime(2025, 1, 1, 9), Decimal('120.00'), 'PLACED', datetime(2025, 1, 1, 9), date(2025, 1, 1)),
        (102, 2, datetime(2025, 1, 1, 11), Decimal('80.00'), 'PLACED', datetime(2025, 1, 1, 11),
            date(2025, 1, 1)),
        (103, 1, datetime(2025, 1, 2, 10), Decimal('200.00'), 'PAID', datetime(2025, 1, 2, 12), date(2025, 1, 2)),
        (104, 3, datetime(2025, 1, 2, 15), Decimal('50.00'), 'PLACED', datetime(2025, 1, 2, 15),
            date(2025, 1, 2)),
        (105, 2, datetime(2025, 1, 3, 8), Decimal('300.00'), 'PAID', datetime(2025, 1, 3, 10), date(2025, 1, 3)),
        (106, 4, datetime(2025, 1, 3, 18), Decimal('150.00'), 'PLACED', datetime(2025, 1, 3, 18),
            date(2025, 1, 3)),
    ],
    """
        order_id INT,
        customer_id INT,
        order_ts TIMESTAMP,
        amount DECIMAL(12,2),
        status STRING,
        updated_at TIMESTAMP,
        order_date DATE
    """,
)


In [ ]:
# Choose the partitioned source directory.
source_path = root + '/orders/partitioned'

staged_orders_df.write.mode('errorifexists').partitionBy('order_date').parquet(source_path)


In [ ]:
# Make these rows available to Spark SQL.
staged_orders_df.createOrReplaceTempView('staged_orders')


In [ ]:
# Register the external source files.
spark.sql(f"""
    CREATE EXTERNAL TABLE spark_catalog.{ns}.orders (
    order_id INT, customer_id INT, order_ts TIMESTAMP, amount DECIMAL(12,2),
    status STRING, updated_at TIMESTAMP
    )
    PARTITIONED BY (order_date DATE)
    STORED AS PARQUET
    LOCATION '{source_path}'
""")


In [ ]:
# Register the source partition directories.
spark.sql(f"""
    MSCK REPAIR TABLE spark_catalog.{ns}.orders
""")


In [ ]:
# Six source orders registered.
assert spark.table(f'spark_catalog.{ns}.orders').count() == 6, 'Six source orders registered'

print('PASS: Six source orders registered')


## Build an independently owned maintenance table
Rewrite the HDFS source into Polaris. This table does not share files with a Hive or migration-snapshot table.
Append one order at a time to create small files and multiple snapshots deliberately.


In [ ]:
# Create the table for this step.
spark.sql(f"""
    CREATE TABLE polaris.{ns}.orders (
    order_id INT, customer_id INT, order_ts TIMESTAMP, amount DECIMAL(12,2),
    status STRING, updated_at TIMESTAMP, order_date DATE
    )
    USING iceberg
    TBLPROPERTIES ('format-version'='2', 'write.metadata.delete-after-commit.enabled'='true', 'write.metadata.previous-versions-max'='2')
""").show(truncate=False)


In [ ]:
# Write this batch into the destination table.
spark.sql(f"""
    INSERT INTO polaris.{ns}.orders SELECT *
    FROM spark_catalog.{ns}.orders
    WHERE order_id=101
""")


In [ ]:
# Write this batch into the destination table.
spark.sql(f"""
    INSERT INTO polaris.{ns}.orders SELECT *
    FROM spark_catalog.{ns}.orders
    WHERE order_id=102
""")


In [ ]:
# Write this batch into the destination table.
spark.sql(f"""
    INSERT INTO polaris.{ns}.orders SELECT *
    FROM spark_catalog.{ns}.orders
    WHERE order_id=103
""")


In [ ]:
# Write this batch into the destination table.
spark.sql(f"""
    INSERT INTO polaris.{ns}.orders SELECT *
    FROM spark_catalog.{ns}.orders
    WHERE order_id=104
""")


In [ ]:
# Write this batch into the destination table.
spark.sql(f"""
    INSERT INTO polaris.{ns}.orders SELECT *
    FROM spark_catalog.{ns}.orders
    WHERE order_id=105
""")


In [ ]:
# Write this batch into the destination table.
spark.sql(f"""
    INSERT INTO polaris.{ns}.orders SELECT *
    FROM spark_catalog.{ns}.orders
    WHERE order_id=106
""")


In [ ]:
# Six independently written orders.
assert spark.table(f'polaris.{ns}.orders').count() == 6, 'Six independently written orders'

print('PASS: Six independently written orders')


In [ ]:
# Read the rows or metadata needed for this check.
base_snapshot = spark.sql(f"""
    SELECT snapshot_id
    FROM polaris.{ns}.orders.snapshots
    ORDER BY committed_at DESC
""").first()[0]


In [ ]:
# Apply the table metadata change.
spark.sql(f"""
    ALTER TABLE polaris.{ns}.orders CREATE TAG before_cutover AS OF VERSION {base_snapshot}
""")


In [ ]:
# Apply the table metadata change.
spark.sql(f"""
    ALTER TABLE polaris.{ns}.orders CREATE BRANCH migration_check AS OF VERSION {base_snapshot}
""")


In [ ]:
# Write this batch into the destination table.
spark.sql(f"""
    INSERT INTO polaris.{ns}.orders.branch_migration_check
    VALUES
    (107, 5, TIMESTAMP '2025-01-01 16:00:00', 75.00, 'PLACED', TIMESTAMP '2025-01-05 09:00:00', DATE '2025-01-01')
""")


In [ ]:
# Main is unchanged during branch validation.
assert spark.table(f'polaris.{ns}.orders').count() == 6, 'Main is unchanged during branch validation'

print('PASS: Main is unchanged during branch validation')


In [ ]:
# Validation branch has seven rows.
assert spark.table(f'polaris.{ns}.orders.branch_migration_check').count() == 7, 'Validation branch has seven rows'

print('PASS: Validation branch has seven rows')


In [ ]:
# Run the Iceberg procedure for this step.
spark.sql(f"""
    CALL polaris.system.fast_forward('{ns}.orders', 'main', 'migration_check')
""").show(truncate=False)


In [ ]:
# Validated branch promoted to main.
assert spark.table(f'polaris.{ns}.orders').count() == 7, 'Validated branch promoted to main'

print('PASS: Validated branch promoted to main')


In [ ]:
# Read the rows or metadata needed for this check.
promoted_snapshot = spark.sql(f"""
    SELECT snapshot_id
    FROM polaris.{ns}.orders.refs
    WHERE name='main'
""").first()[0]


In [ ]:
# Run the Iceberg procedure for this step.
spark.sql(f"""
    CALL polaris.system.set_current_snapshot(table => '{ns}.orders', ref => 'before_cutover')
""")


In [ ]:
# Tag restores the cutover baseline.
assert spark.table(f'polaris.{ns}.orders').count() == 6, 'Tag restores the cutover baseline'

print('PASS: Tag restores the cutover baseline')


In [ ]:
# Run the Iceberg procedure for this step.
spark.sql(f"""
    CALL polaris.system.set_current_snapshot(table => '{ns}.orders', snapshot_id => {promoted_snapshot})
""")


In [ ]:
# Validated state restored after rollback rehearsal.
assert spark.table(f'polaris.{ns}.orders').count() == 7, 'Validated state restored after rollback rehearsal'

print('PASS: Validated state restored after rollback rehearsal')


## Measure housekeeping effects
Small files increase file opens and metadata planning work. Compaction trades rewrite I/O now for fewer files in later scans.
Record active file count, bytes, manifest count, query plan and elapsed time. Elapsed time on seven rows is illustrative;
cache, startup and scheduling dominate, so it is not a benchmark or a guaranteed speedup.
Snapshot expiration reclaims obsolete history; it does not combine active data files.
Manifest rewriting can reduce planning overhead. Metadata and orphan cleanup control storage accumulation;
neither operation reorganizes the active rows or guarantees faster queries.


In [ ]:
# Import the module used in the next step.
import time


In [ ]:
# Measure this query execution.
start_time = time.perf_counter()

total = spark.sql(f"""
    SELECT sum(amount)
    FROM polaris.{ns}.orders
""").first()[0]

elapsed = time.perf_counter() - start_time


In [ ]:
# Read the rows or metadata needed for this check.
stats = spark.sql(f"""
    SELECT count(*) AS files, sum(file_size_in_bytes) AS bytes
    FROM polaris.{ns}.orders.files
""").first()


In [ ]:
# Read the rows or metadata needed for this check.
manifests = spark.sql(f"""
    SELECT count(*)
    FROM polaris.{ns}.orders.manifests
""").first()[0]


In [ ]:
# Record the file and query measurements before housekeeping.
before = dict(files=stats.files, bytes=stats.bytes, manifests=manifests, total=str(total), seconds=round(elapsed, 3))


In [ ]:
# Inspect how Spark plans this query.
spark.sql(f"""
    EXPLAIN FORMATTED SELECT sum(amount)
    FROM polaris.{ns}.orders
""").show(truncate=False)


In [ ]:
# Run the Iceberg procedure for this step.
spark.sql(f"""
    CALL polaris.system.rewrite_data_files(
    table => '{ns}.orders', options => map('rewrite-all', 'true', 'target-file-size-bytes', '134217728')
    )
""").show(truncate=False)


In [ ]:
# Run the Iceberg procedure for this step.
spark.sql(f"""
    CALL polaris.system.rewrite_manifests('{ns}.orders')
""").show(truncate=False)


In [ ]:
# Measure this query execution.
start_time = time.perf_counter()

total = spark.sql(f"""
    SELECT sum(amount)
    FROM polaris.{ns}.orders
""").first()[0]

elapsed = time.perf_counter() - start_time


In [ ]:
# Read the rows or metadata needed for this check.
stats = spark.sql(f"""
    SELECT count(*) AS files, sum(file_size_in_bytes) AS bytes
    FROM polaris.{ns}.orders.files
""").first()


In [ ]:
# Read the rows or metadata needed for this check.
manifests = spark.sql(f"""
    SELECT count(*)
    FROM polaris.{ns}.orders.manifests
""").first()[0]


In [ ]:
# Record the file and query measurements after housekeeping.
after = dict(files=stats.files, bytes=stats.bytes, manifests=manifests, total=str(total), seconds=round(elapsed, 3))


In [ ]:
# Display the checked result.
print('Before:', before)


In [ ]:
# Display the checked result.
print('After: ', after)


In [ ]:
# Compaction reduced active data file count.
assert after['files'] < before['files'], 'Compaction reduced active data file count'

print('PASS: Compaction reduced active data file count')


In [ ]:
# Compaction preserved the amount total.
assert after['total'] == before['total'] == '975.00', 'Compaction preserved the amount total'

print('PASS: Compaction preserved the amount total')


In [ ]:
# Compaction preserved all rows.
assert spark.table(f'polaris.{ns}.orders').count() == 7, 'Compaction preserved all rows'

print('PASS: Compaction preserved all rows')


## Expire snapshots while a tag protects history
Snapshot expiration removes old snapshot references and files no longer needed by retained history.
Branches and tags can keep old snapshots and files alive. A table is not a backup of itself: preserve catalog and storage backups separately.
For this disposable table only, use a cutoff captured after the writes. Production retention must exceed the longest reader,
writer, recovery and audit requirements. Do not apply this aggressive example to shared-file migration tables.


In [ ]:
# Apply the table metadata change.
spark.sql(f"""
    ALTER TABLE polaris.{ns}.orders DROP BRANCH migration_check
""")


In [ ]:
# Read the rows or metadata needed for this check.
cutoff = spark.sql("""
    SELECT date_format(current_timestamp(), 'yyyy-MM-dd HH:mm:ss.SSS')
""").first()[0]


In [ ]:
# Run the Iceberg procedure for this step.
spark.sql(f"""
    CALL polaris.system.expire_snapshots(
    table => '{ns}.orders', older_than => TIMESTAMP '{cutoff}', retain_last => 1
    )
""").show(truncate=False)


In [ ]:
# Tag-protected snapshot remains readable.
assert spark.sql(f"""
    SELECT count(*)
    FROM polaris.{ns}.orders VERSION AS OF {base_snapshot}
""").first()[0] == 6, 'Tag-protected snapshot remains readable'

print('PASS: Tag-protected snapshot remains readable')


In [ ]:
# Apply the table metadata change.
spark.sql(f"""
    ALTER TABLE polaris.{ns}.orders DROP TAG before_cutover
""")


In [ ]:
# Read the rows or metadata needed for this check.
cutoff = spark.sql("""
    SELECT date_format(current_timestamp(), 'yyyy-MM-dd HH:mm:ss.SSS')
""").first()[0]


In [ ]:
# Run the Iceberg procedure for this step.
spark.sql(f"""
    CALL polaris.system.expire_snapshots(
    table => '{ns}.orders', older_than => TIMESTAMP '{cutoff}', retain_last => 1
    )
""").show(truncate=False)


In [ ]:
# Only the current snapshot remains after releasing the tag.
assert spark.sql(f"""
    SELECT count(*)
    FROM polaris.{ns}.orders.snapshots
""").first()[0] == 1, 'Only the current snapshot remains after releasing the tag'

print('PASS: Only the current snapshot remains after releasing the tag')


In [ ]:
# Load the Iceberg table to inspect its metadata and file I/O.
iceberg_table = spark._jvm.org.apache.iceberg.spark.Spark3Util.loadIcebergTable(spark._jsparkSession, f'polaris.{ns}.orders')


In [ ]:
# All current data files still exist.
assert all((iceberg_table.io().newInputFile(p).exists() for p in {file_row.file_path for file_row in spark.sql(f"""
    SELECT file_path
    FROM polaris.{ns}.orders.files
""").collect()})), 'All current data files still exist'

print('PASS: All current data files still exist')


## Discover and remove one known orphan
An orphan is a file unreferenced by table metadata, not simply an old file. Create a single dummy file in a dedicated HDFS
directory inside an independently owned HDFS Iceberg table and make its modification time four days old. The cleanup scope is that directory only.
Use the default three-day safety interval, inspect the dry run, then delete the known candidate. Never shorten the interval
to clean up recent writes: a concurrent writer may not have committed its files yet.


In [ ]:
# Create the table for this step.
spark.sql(f"""
    CREATE TABLE lake.{ns}.orphan_demo (id INT)
    USING iceberg
""")


In [ ]:
# Write this batch into the destination table.
spark.sql(f"""
    INSERT INTO lake.{ns}.orphan_demo
    VALUES (1)
""")


In [ ]:
# Find the location owned by the HDFS demonstration table.
orphan_table = spark._jvm.org.apache.iceberg.spark.Spark3Util.loadIcebergTable(spark._jsparkSession, f'lake.{ns}.orphan_demo')


In [ ]:
# Limit cleanup to a dedicated directory inside the demonstration table.
orphan_dir = orphan_table.location() + '/orphan-demo'


In [ ]:
# Name the one uncommitted file used in the cleanup exercise.
orphan_path = orphan_dir + '/uncommitted-demo.txt'


In [ ]:
# Inspect or prepare the file used in this exercise.
jpath = spark._jvm.org.apache.hadoop.fs.Path(orphan_path)


In [ ]:
# Use the filesystem that owns this path.
fs = jpath.getFileSystem(spark._jsc.hadoopConfiguration())


In [ ]:
# Inspect or prepare the file used in this exercise.
fs.mkdirs(jpath.getParent())


In [ ]:
# Inspect or prepare the file used in this exercise.
stream = fs.create(jpath, False)


In [ ]:
# Inspect or prepare the file used in this exercise.
stream.writeUTF('Uncommitted demonstration file')


In [ ]:
# Inspect or prepare the file used in this exercise.
stream.close()


In [ ]:
# Inspect or prepare the file used in this exercise.
fs.setTimes(jpath, int((time.time() - 4 * 86400) * 1000), -1)


In [ ]:
# Run the Iceberg procedure for this step.
dry = spark.sql(f"""
    CALL lake.system.remove_orphan_files(
    table => '{ns}.orphan_demo',
    location => '{orphan_dir}', dry_run => true
    )
""").collect()


In [ ]:
# Dry run found only the deliberately created orphan.
assert len(dry) == 1 and dry[0][0] == orphan_path, 'Dry run found only the deliberately created orphan'

print('PASS: Dry run found only the deliberately created orphan')


In [ ]:
# Dry run retained the candidate.
assert hdfs.exists(HadoopPath(orphan_path)), 'Dry run retained the candidate'

print('PASS: Dry run retained the candidate')


In [ ]:
# Run the Iceberg procedure for this step.
spark.sql(f"""
    CALL lake.system.remove_orphan_files(
    table => '{ns}.orphan_demo',
    location => '{orphan_dir}', dry_run => false
    )
""").show(truncate=False)


In [ ]:
# Approved orphan candidate removed.
assert not hdfs.exists(HadoopPath(orphan_path)), 'Approved orphan candidate removed'

print('PASS: Approved orphan candidate removed')


In [ ]:
# Current HDFS Iceberg data remains readable.
assert spark.table(f'lake.{ns}.orphan_demo').count() == 1, 'Current HDFS Iceberg data remains readable'

print('PASS: Current HDFS Iceberg data remains readable')


## Metadata cleanup and an operating schedule
`write.metadata.delete-after-commit.enabled=true` with `write.metadata.previous-versions-max=2` bounds tracked old metadata JSON files
as new commits occur. It does not compact data, expire snapshots, or discover every untracked file.
Inspect the current metadata log instead of manually deleting metadata files.

| Work | Trigger and check |
|---|---|
| Data-file compaction | Review file counts, size distribution and rewrite cost after ingestion |
| Manifest rewrite | Review manifest count and planning overhead |
| Snapshot expiration | Apply the approved recovery window; review branch and tag retention first |
| Metadata JSON retention | Bound tracked previous versions; verify after commits |
| Orphan cleanup | Dry run with a writer-safe age and narrow table-owned scope |
| Recovery rehearsal | Verify catalog restore, file availability, permissions and batch replay |

Retain history before optimizing storage savings. After expiration, rollback is possible only to a snapshot that still exists.


In [ ]:
# Read the rows or metadata needed for this check.
spark.sql(f"""
    SELECT *
    FROM polaris.{ns}.orders.metadata_log_entries
""").show(truncate=False)


In [ ]:
# Inspect the catalog metadata.
props = {row.key: row.value for row in spark.sql(f"""
    SHOW
    TBLPROPERTIES polaris.{ns}.orders
""").collect()}


In [ ]:
# Automatic tracked metadata cleanup is enabled.
assert props['write.metadata.delete-after-commit.enabled'] == 'true', 'Automatic tracked metadata cleanup is enabled'

print('PASS: Automatic tracked metadata cleanup is enabled')


In [ ]:
# Tracked metadata retention is bounded.
assert props['write.metadata.previous-versions-max'] == '2', 'Tracked metadata retention is bounded'

print('PASS: Tracked metadata retention is bounded')


In [ ]:
# Load the Iceberg table to inspect its metadata and file I/O.
iceberg_table = spark._jvm.org.apache.iceberg.spark.Spark3Util.loadIcebergTable(spark._jsparkSession, f'polaris.{ns}.orders')


In [ ]:
# Read the tracked previous metadata versions.
tracked_metadata = iceberg_table.operations().current().previousFiles()


In [ ]:
# Tracked previous metadata JSON list respects the configured limit.
assert tracked_metadata.size() <= 2, 'Tracked previous metadata JSON list respects the configured limit'

print('PASS: Tracked previous metadata JSON list respects the configured limit')


In [ ]:
# Maintenance left the final table readable.
assert spark.table(f'polaris.{ns}.orders').count() == 7, 'Maintenance left the final table readable'

print('PASS: Maintenance left the final table readable')


## Classroom exercises and assignment

1. Compare file count and elapsed time before and after compaction. State which result is reliable at this scale.
2. Explain why a retained tag can prevent storage reclamation.
3. Assignment: write a monthly maintenance runbook for a table with 48-hour late arrivals and seven-day rollback. Include branch retention, concurrent writers, dry-run review, monitoring and recovery checks.


<details>
    <summary>Review points</summary>

    File-count reduction is deterministic here; timing is not a performance guarantee. Retained references preserve needed history and files. A seven-day rollback promise requires at least that history and file availability, plus operational margin. Orphan safety intervals must also cover long-running writes. Never run aggressive cleanup on shared migration files.

</details>


In [ ]:
# Display the checked result.
print('ALL CORE CHECKS PASSED')


In [ ]:
# Release this notebook Spark session.
spark.stop()
